# Part 1. Prompt Engineering

- A **prompt** is a **set of instructions** (including content   like text, images, etc) provided to an LLM to perform a task.

- **Prompt Engineering** is the process of crafting such the set of instructions that gets an LLM model to generate the desired outcome (i.e., solving the task). 

#### Components of Well-Structured Prompts
- **Role**: The role the LLM should adopt.
- **Task Description**: The specific instruction or question.
- **Context**: Additional information needed for the task.
- **Output Format**: How the response should be structured.
- **Examples (a.k.a. Few-Shot "Learning")** (optional): Sample input/output pairs.

For the "Prompt Engineering and Structured Outputs" section of our tutorial, we will rely on OpenAI Software Development Kit (OpenAI SDK), which can be installed by running `pip install openai` (or similar).
We will use models running within Ollama and OpenAI SDK to connect to Ollama.

In [1]:
# Start our ollama server in the background to host our LLMs
from ollama_utils import start_ollama_server, stop_ollama_server

# start ollama server
start_ollama_server()

# model names
llama8b = "llama3.1:8b"
llama3b = "llama3.2:latest"

🚀 Starting Ollama server...
📄 Server logs: ollama_server.log
📍 API endpoint: http://localhost:11434
⏳ Waiting 5 seconds for server startup...
✅ Ollama server ready! PID: 2269502


In [2]:
# Import OpenAI client class
from openai import OpenAI

# Import other modules
import textwrap

In [3]:
# Connect to Ollama running on the backend

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [4]:
# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI,
        model: str,
        user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        temperature: float = 0.5       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

#### 1. Vague vs. Precise Prompt
Let's see how an LLM's output changes when we start from a vague prompt, then adding a persona/role, and finally specifying the task.

In [5]:
# Vague prompt = vague result.
# We should expect a generic output from the LLM.
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=vague_prompt, 
)

print(vague_response)
# textwrap.fill(vague_response, width=80)

Albert Einstein (1879-1955) was a renowned German-born physicist who revolutionized our understanding of space, time, and gravity. He is widely regarded as one of the most influential scientists of the 20th century.

**Early Life and Education**

Einstein was born in Munich, Germany to Hermann and Pauline Einstein. His early life was marked by curiosity and a passion for learning. He began teaching himself mathematics and physics at an early age and developed a strong interest in philosophy. Einstein's family moved to Switzerland when he was 10 years old, where he attended school and later enrolled in the Swiss Federal Polytechnic University.

**Career**

Einstein graduated from university in 1900 with a degree in physics and began working as a patent clerk in Bern, Switzerland. During this time, he developed his famous theory of special relativity (1905), which posited that time and space are relative and can be affected by gravity. This groundbreaking work introduced the concept of t

In [6]:
# Add role (persona) to the vague prompt.
# The tone of the output should match the role provided in the prompt.
vague_system_prompt_with_persona = "You are a high-school physics teacher."
vague_user_prompt_with_persona = "Tell me about Albert Einstein."


response_with_persona = generate_response(
    client=openai_client,
    model=llama8b,
    system_prompt=vague_system_prompt_with_persona,
    user_prompt=vague_user_prompt_with_persona,
)

print(response_with_persona)
# textwrap.fill(response_with_persona, width=80)

Albert Einstein! One of the most brilliant minds in the history of science. I just love teaching my students about him.

Born on March 14, 1879, in Ulm, Germany, Einstein was a theoretical physicist who revolutionized our understanding of space and time. He's best known for his theory of relativity, which challenged the long-held notion that time and space are fixed and absolute.

Einstein's work built upon the ideas of Sir Isaac Newton, but he took it to a whole new level. His special theory of relativity, introduced in 1905, showed that the laws of physics are the same for all observers in uniform motion relative to one another. This meant that time dilation occurs when objects move at high speeds – time appears to slow down for an observer watching from a stationary frame.

But that was just the beginning! In his theory of general relativity (1915), Einstein introduced the concept of gravity as curvature of spacetime caused by massive objects. Think about it: gravity isn't a force t

In [7]:
# Finally let's narrow down the scope of the output by providing 
# specific task and the output format.
precise_system_prompt = "You are a high-school physics teacher."

precise_user_prompt = """
Explain the role Albert Einstein played in the development of physics 
in 20th century.
Write two paragraphs.
"""

precise_response = generate_response(
    client=openai_client,
    model=llama8b,
    system_prompt=precise_system_prompt,
    user_prompt=precise_user_prompt
)

print(precise_response)
# textwrap.fill(precise_response, width=80)

Albert Einstein's impact on the development of physics in the 20th century was profound and far-reaching. His theory of special relativity, introduced in 1905, revolutionized our understanding of space and time. By postulating that the laws of physics are the same for all observers in uniform motion relative to one another, Einstein challenged the long-held notion of absolute time and space. This led to a fundamental shift in our comprehension of the universe, where time and space became intertwined as a single entity known as spacetime.

Einstein's subsequent theory of general relativity, introduced in 1915, further transformed our understanding of gravity and its effects on spacetime. He proposed that gravity is not a force, but rather the curvature of spacetime caused by massive objects. This idea led to a new understanding of gravitational waves, black holes, and the behavior of celestial bodies. Einstein's work laid the foundation for many subsequent breakthroughs in modern physic

In the case of the vague prompt, we can see that the output text covers various aspects of Albert Einstein's live without providing depth into any of those. Advancing forward, when we provide a role ("a high-school physics teacher"), the output changes its format and is "tailored" to the role/persona specified. 
Finally, with the precise prompt, we **guide** an LLM to produce a specific but more detailed response. While for some cases (e.g., famous personas) we might want to learn both perspectives (in breadth and depth), for AI applications we typically want to narrow down the scope of the task to get more specific outputs.

So, **Best Practice #1: Be specific and direct.**

#### 2. Adding Context
Any LLM model "knows" only what it was trained on.
However, you can incorporate new information by providing 
it as a context.

In [8]:
# Source: https://tacc.utexas.edu/about/staff-directory/niall-gaffney/
context = """
NIALL GAFFNEY

Data and AI Directorate

Phone: 512-475-9504 | Email: ngaffney@tacc.utexas.edu

Education: B.A., M.A., Ph.D., Astronomy University of Texas at Austin

Niall Gaffney's background primarily revolves around the management and utilization of 
large inhomogeneous scientific datasets. Niall, who earned his B.A., M.A., and Ph.D. 
degrees in astronomy from The University of Texas at Austin, joined TACC in May 2013. 
Most of his focus has been on creating environments to foster better data practices 
from improving metadata, data processing, analysis, and reuse. He focuses on improving 
researchers' data practices to accelerate outcomes and better feed the Machine Learning 
and Artificial Intelligence applications which are becoming more broadly adopted in 
science and engineering research fields. Much of this stems from his 13 years as designer 
and developer for the archives at the Space Telescope Science Institute (STScI), which 
holds the data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope 
missions.

He was also a leader in developing the Hubble Legacy Archive. This project harvested 
the 20+ years of Hubble Space Telescope data to create some of the most sensitive 
astronomical data products available for open research. Before his work at STScI, 
Niall was worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) 
project at the McDonald Observatory in west Texas. This was the start of his work in 
planning experiments and then cataloging the data the HET produced.
"""

**Temperature** is the parameter for controlling the randomness (or creativity) of a model. By increasing the temperature from 0 to 1, we're increasing the randomness (hence more creativity) of the response. 

In [9]:
# First let's see what a model "knows" about Niall Gaffney.
# In other words, do NOT provide any context first. 

# Tip: Modify temperature and see how this parameter influences
# the output of the model by sending requests multiple times.
no_context_prompt = "Who is Niall Gaffney?"

response_wo_context = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=no_context_prompt,
    temperature=0
)

print(response_wo_context)

Niall Gaffney is an Irish computer scientist and engineer who has made significant contributions to the field of data management and storage systems. He is currently the Director of IBM Research Ireland, where he leads a team focused on developing innovative technologies for data management, artificial intelligence, and cybersecurity.

Gaffney has held various leadership positions within IBM, including serving as the Chief Technology Officer (CTO) for IBM's Cloud Storage Services. He has also been involved in several high-profile research projects, including the development of IBM's Cloud Object Store and the company's efforts to advance the field of data storage and management.

Gaffney is widely recognized as a thought leader in his field, and he has published numerous papers on topics related to data management, cloud computing, and artificial intelligence. He holds several patents in these areas and has received various awards for his contributions to technology innovation.

Would 

In [10]:
# Now, let's add the context.
# Tip: run this part multiple times for temperature=0.0 and temperature=1.0
prompt_with_context = no_context_prompt + f"\nContext: {context}"

response_with_context = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_with_context,
    temperature=0.0
)

print(response_with_context)

Niall Gaffney is a researcher with a background in astronomy, currently working as part of the Data and AI Directorate at TACC (Texas Advanced Computing Center). He has a strong focus on managing and utilizing large scientific datasets, improving data practices, and accelerating research outcomes through Machine Learning and Artificial Intelligence applications.

Specifically, his experience includes:

* 13 years as designer and developer for the archives at the Space Telescope Science Institute (STScI), where he worked with data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope missions.
* Leadership in developing the Hubble Legacy Archive, which harvested 20+ years of Hubble Space Telescope data to create sensitive astronomical data products available for open research.
* Previous work as "the friend of the telescope" for the Hobby Eberly Telescope (HET) project at the McDonald Observatory in west Texas, where he planned experiments and cataloged data.

He holds

We've seen that by providing context, we can "add" new information/knowledge to a model without retraining it. What's more, by providing context we can also reduce the occurrence of hallucinations [1]. However, note that adding the context **does NOT guarantee** that a model will strictly follow it [1].

Also, adding additional instructions like "generate response based on the context provided" to the prompt can also be helpful.

**Best Practice #2: Add specific context to your prompts when applicable.**

[1] Huyen, C. (2024). Prompt Engineering. In AI Engineering: Building Applications with Foundation Models. (pp. 211-252) O'Reilly Media, Inc.

#### 3. Prompt Chaining
Break complex tasks into simpler subtasks.
Use each LLM's output as a **context** for the next prompt/step.

In [11]:

topic = "the impact of Albert Einstein on philosophy of 20th century"

# Step 1: Create an outline for the blog post
prompt_step_1 = f"Come up with a 3 point outline for a post about: {topic}"

response_step_1 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_1,
    temperature=0.7
)

print(response_step_1)

Here's a 3-point outline for a post on the impact of Albert Einstein on the philosophy of the 20th century:

**Title:** "The Theory of Relativity and Beyond: Albert Einstein's Impact on 20th Century Philosophy"

**I. Challenging Traditional Notions of Space, Time, and Causality**

* Describe how Einstein's theory of relativity revolutionized our understanding of space and time, challenging Newtonian notions of absolute space and time.
* Discuss how this challenged traditional philosophical concepts such as determinism and causality, raising questions about the nature of free will and the role of randomness in the universe.

**II. Influencing Philosophers' Views on Knowledge and Reality**

* Outline how Einstein's work influenced philosophers like Henri Bergson, Martin Heidegger, and Jean-Paul Sartre to rethink their views on knowledge, reality, and human experience.
* Discuss how physicists like Werner Heisenberg and Niels Bohr also influenced philosophical debates about the nature of 

In [12]:
# Step 2: Write introduction using the outline
prompt_step_2=f"""
Using the following OUTLINE, write an introduction paragraph with 80-100 words.
OUTLINE: {response_step_1}
Hook the reader with a surprising fact in the first sentence.
"""

response_step_2 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_2,
    temperature=0.7
)

print(response_step_2)

Here is an introduction paragraph for "The Theory of Relativity and Beyond: Albert Einstein's Impact on 20th Century Philosophy":

Did you know that Albert Einstein's theory of relativity wasn't just a groundbreaking scientific concept, but also a philosophical game-changer? By introducing the idea that space and time are not fixed or absolute, but rather relative and interconnected, Einstein's work had far-reaching implications for philosophers' understanding of causality, free will, and the nature of reality. As this post will explore, Einstein's theory of relativity sparked a ripple effect in 20th century philosophy, influencing some of the most prominent thinkers of the time, from Henri Bergson to Jean-Paul Sartre, and even inspiring new directions in philosophical thought, including hermitianism, emergentism, and postmodernism.


In [13]:
# Step 3: Come up with a few options for the title
prompt_step_3 = f"""
Based on the INTRODUCTION below, come up with 3 catchy blog post titles.
INTRODUCTION: {response_step_2}
Format your output as 3 bullet points.
""" 
response_step_3 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_3,
    temperature=1
)

print(response_step_3)

Here are three catchy blog post title options based on the introduction:

• "The Ripple Effect: How Einstein's Theory of Relativity Shaped 20th Century Philosophy"
• "Einstein's Mind-Bending Idea: The Far-Reaching Impact on Philosophical Thought"
• "Beyond Space and Time: Einstein's Theory of Relativity and the Birth of a New Era in Philosophy"


Main motivation behind the prompt chaining technique: a few smaller prompts are better than one that is a giant one. If your application require solving complex tasks with multiple steps, divide them into smaller subtasks by introducing smaller prompts and chain LLM's outputs together with the following prompts.

Remember: When working with LLMs, simpler instructions are better than complex ones.

**Best Practice #3: Break complex prompts into smaller ones.**

#### 4. More Practice - Code Generation

In [14]:
role = "a Python Developer"
task_description = "to develop Python code based on the specification provided"
context = "write a Python function that generates Fibonacci sequence; use a `while` loop"
output_format = """\n
    **EXPLANATION**: [provide a brief explanation of your solution]
    **PYTHON CODE**: [a block of Python code]
"""

# Do not provide additional text after the **PYTHON CODE** section

In [15]:
system_prompt = f"You are {role}. Your task is {task_description}."
user_prompt = f"""
Here are additional instructions: {context}.
OUTPUT FORMAT: {output_format}
"""

print("SYSTEM PROMPT:", system_prompt, sep="\n")
print("-" * 100)
print("USER PROMPT:", user_prompt, sep="\n")

SYSTEM PROMPT:
You are a Python Developer. Your task is to develop Python code based on the specification provided.
----------------------------------------------------------------------------------------------------
USER PROMPT:

Here are additional instructions: write a Python function that generates Fibonacci sequence; use a `while` loop.
OUTPUT FORMAT: 

    **EXPLANATION**: [provide a brief explanation of your solution]
    **PYTHON CODE**: [a block of Python code]




In [16]:
response = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

print(response)

**EXPLANATION**: 
The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, usually starting with 0 and 1. This function generates the Fibonacci sequence up to a given number of terms.

**PYTHON CODE**:

```python
def fibonacci(n):
    """
    Generate Fibonacci sequence up to n terms.

    Args:
        n (int): Number of terms in the sequence.

    Returns:
        list: The generated Fibonacci sequence.
    """

    # Initialize the first two numbers in the sequence
    a, b = 0, 1

    # Create an empty list to store the sequence
    fib_sequence = [a, b]

    # Use a while loop to generate the rest of the sequence
    i = 2
    while len(fib_sequence) < n:
        # Calculate the next number in the sequence as the sum of the previous two
        a, b = b, a + b

        # Append the new number to the sequence list
        fib_sequence.append(b)

    return fib_sequence


# Example usage: print the first 10 numbers in the Fibonacci sequen

In [17]:
stop_ollama_server() 

🛑 Ollama server stopped
